# Split `enriched_moves.parquet` into train and test, by user


- No user is split
- 80/20 by moves (seed 101)

Outputs:

| File | Contents |
|---|---|
| `enriched_moves_train.parquet` | every point of the training users |
| `enriched_moves_test.parquet` | every point of the test users |
| `test_trips.csv` | `uid, tid` of every test trip
| `split_info.json` | settings and counts |


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

TEST_FRAC = 0.2  
SEED = 101       

# URBAN-SYN root: the first folder at or above the notebook's folder that holds the moves file.
MOVES_REL = Path("data/nyc_output_tabular/output/enriched_moves.parquet")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / MOVES_REL).exists()), None)
assert ROOT is not None, f"{MOVES_REL} not found in {Path.cwd()} or above: open the notebook inside URBAN-SYN, or set ROOT by hand"

MOVES = ROOT / MOVES_REL
OUT_DIR = ROOT / "data/split_by_user"

## Load the moves

In [2]:
moves = pd.read_parquet(MOVES)
print(f"{len(moves):,} points")
print("index  :", list(moves.index.names))
print("columns:", list(moves.columns))

flat = moves.reset_index() if any(name is not None for name in moves.index.names) else moves.reset_index(drop=True)
KEYS = ["uid", "tid", "move_id"]          # one trajectory (move) = one (uid, tid, move_id)
missing = [k for k in KEYS if k not in flat.columns]
assert not missing, f"missing columns: {missing}"

per_user = pd.DataFrame({
    "moves": flat.drop_duplicates(KEYS).groupby("uid").size(),
    "points": flat.groupby("uid").size(),
})
print(f"{len(per_user):,} users, {per_user['moves'].sum():,} moves")
print(f"trips (tid) per user: {flat.groupby('uid')['tid'].nunique().describe()[['mean', 'max']].round(2).to_dict()}")
print(f"moves per user      : {per_user['moves'].describe()[['mean', '50%', 'max']].round(1).to_dict()}")
moves.head()

6,623,956 points
index  : ['tid', 'move_id']
columns: ['datetime', 'uid', 'lat', 'lng', 'label']
18,762 users, 19,474 moves
trips (tid) per user: {'mean': 1.0, 'max': 1.0}
moves per user      : {'mean': 1.0, '50%': 1.0, 'max': 28.0}


datetime  uid        lat        lng  label
tid move_id                                                      
3   0       2021-05-23 08:23:21    3  40.821955 -74.139058    2.0
    0       2021-05-23 08:23:24    3  40.822035 -74.139246    2.0
    0       2021-05-23 08:23:26    3  40.822115 -74.139433    2.0
    0       2021-05-23 08:23:28    3  40.822036 -74.139485    2.0
    0       2021-05-23 08:23:32    3  40.821987 -74.139464    2.0

## Choose the test users

Shuffle the users with the seed, then add them to the test set one by one, in that order, and stop at the
number of users whose moves come closest to 20% of all moves.

In [3]:
rng = np.random.default_rng(SEED)
shuffled = per_user.iloc[rng.permutation(len(per_user))]
cum_moves = shuffled["moves"].cumsum().to_numpy()
target = TEST_FRAC * per_user["moves"].sum()
n_test_users = int(np.argmin(np.abs(cum_moves - target))) + 1   # closest to the target

test_users = set(shuffled.index[:n_test_users])
is_test = flat["uid"].isin(test_users).to_numpy()   # one value per row of moves

print(f"test users : {n_test_users:,} of {len(per_user):,} ({n_test_users / len(per_user):.1%})")
print(f"test moves : {int(cum_moves[n_test_users - 1]):,} of {per_user['moves'].sum():,} "
      f"({cum_moves[n_test_users - 1] / per_user['moves'].sum():.1%}; target {TEST_FRAC:.0%})")

test users : 3,755 of 18,762 (20.0%)
test moves : 3,895 of 19,474 (20.0%; target 20%)


## Create the files

In [4]:
train = moves[~is_test]   # same index, columns and row order as the original
test = moves[is_test]

OUT_DIR.mkdir(parents=True, exist_ok=True)
train.to_parquet(OUT_DIR / "enriched_moves_train.parquet")
test.to_parquet(OUT_DIR / "enriched_moves_test.parquet")

# uid, tid of every test trip (the format of fmm_scripts/nyc_test.csv)
test_trips = flat.loc[is_test, ["uid", "tid"]].drop_duplicates().sort_values(["uid", "tid"])
test_trips.to_csv(OUT_DIR / "test_trips.csv", index=False)

def counts(rows):
    return {"users": int(rows["uid"].nunique()),
            "moves": int(len(rows.drop_duplicates(KEYS))),
            "points": int(len(rows))}

info = {
    "source": str(MOVES_REL),
    "split_by": "uid (every move of a user on the same side)",
    "test_frac_target": TEST_FRAC,
    "seed": SEED,
    "train": counts(flat[~is_test]),
    "test": counts(flat[is_test]),
}
(OUT_DIR / "split_info.json").write_text(json.dumps(info, indent=2))
for name in ["enriched_moves_train.parquet", "enriched_moves_test.parquet", "test_trips.csv", "split_info.json"]:
    print(f"{name:30s} {(OUT_DIR / name).stat().st_size / 1e6:8.1f} MB")

enriched_moves_train.parquet      122.4 MB
enriched_moves_test.parquet        30.6 MB
test_trips.csv                      0.1 MB
split_info.json                     0.0 MB


## Test

In [5]:
train_flat, test_flat = flat[~is_test], flat[is_test]

# No user (and so no trip or move) on both sides
assert not set(train_flat["uid"]) & set(test_flat["uid"]), "a user is in both sets"
assert not (set(map(tuple, train_flat[KEYS].drop_duplicates().to_numpy()))
            & set(map(tuple, test_flat[KEYS].drop_duplicates().to_numpy()))), "a move is in both sets"

# Every point exactly once, and the files read back exactly as written
assert len(train) + len(test) == len(moves)
pd.testing.assert_frame_equal(pd.read_parquet(OUT_DIR / "enriched_moves_train.parquet"), train)
pd.testing.assert_frame_equal(pd.read_parquet(OUT_DIR / "enriched_moves_test.parquet"), test)
assert len(test_trips) == len(test_flat[["uid", "tid"]].drop_duplicates())
print("all checks passed")

summary = pd.DataFrame({"train": info["train"], "test": info["test"]}).T
for col in ["users", "moves", "points"]:
    summary[f"{col} %"] = (100 * summary[col] / summary[col].sum()).round(1)
summary

all checks passed


,users,moves,points,users %,moves %,points %
train,15007,15579,5318335,80.0,80.0,80.3
test,3755,3895,1305621,20.0,20.0,19.7


Transport-mode share of the moves in each set 

In [6]:
MODE_NAMES = {0: "walk", 1: "bike", 2: "bus", 3: "car", 4: "subway", 5: "train", 6: "taxi"}
if "label" in flat.columns:
    per_move = flat.drop_duplicates(KEYS).copy()   # label is the same on every point of a move
    per_move["split"] = np.where(per_move["uid"].isin(test_users), "test", "train")
    modes =(pd.crosstab(per_move["label"].map(lambda v: MODE_NAMES.get(v, v)), per_move["split"], normalize="columns") * 100).round(1)
    display(modes[["train", "test"]])
else:
    print("no label column")

split,train,test
label,,
bike,5.8,6.1
bus,58.5,58.4
car,15.1,15.0
subway,0.6,0.6
train,0.0,0.1
walk,20.0,19.8


## Using the split in the baselines

- **Baselines that read separate train and test files:** point them at `enriched_moves_train.parquet` and
  `enriched_moves_test.parquet`. They have the same layout as `enriched_moves.parquet`, so they load the same
  way.
- **Baselines that read the full file plus a test list** (DiffTraj's `prepare_data.py --test-csv`, the
  Diff-RNTraj converter's `--test-trips`): keep
  `enriched_moves.parquet` and pass `test_trips.csv` as the test list. It has the same `uid, tid` columns.